<a href="https://colab.research.google.com/github/jppeirce/DSC210-Foundations-of-Data-Science/blob/main/ClassNotebooks/09-unsupervised_learning/09b-penguin_clustering_case_study.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 9 Case Study: Species, Sex, or Size?

**DSC 210 Foundations of Data Science**

*A complete clustering analysis of the Palmer penguins, using every method from Module 9.*

```
[ ASK ]  ->  [ GET ]  ->  [ EXPLORE ]  ->  [ MODEL ]  ->  [ COMMUNICATE ]
```

References:
- Module 9 course notebook, Sections 2 to 8 (k-means, choosing k, hierarchical clustering, DBSCAN, PCA)
- Huber, *Hands-on Introduction to Data Science with Python* (CC BY-NC-SA 4.0): [Clustering](https://florian-huber.github.io/data_science_course/notebooks/12_clustering.html) and [Clustering beyond the basics](https://florian-huber.github.io/data_science_course/notebooks/12b_clustering_beyond_the_basics.html) (Ward linkage)
- Gorman, K. B., Williams, T. D., and Fraser, W. R. (2014). Ecological sexual dimorphism and environmental variability within a community of Antarctic penguins (genus *Pygoscelis*). *PLoS ONE* 9(3): e90081. https://doi.org/10.1371/journal.pone.0090081
- Horst, A. M., Hill, A. P., and Gorman, K. B. (2020). *palmerpenguins: Palmer Archipelago (Antarctica) penguin data*. https://allisonhorst.github.io/palmerpenguins/

*Last major revision: 2026-10-26*

## Key Concepts

- Carry a clustering analysis through the whole pipeline, from a question to a written conclusion
- Read a crosstab of clusters against hidden labels, and explain the *mistakes*, not only the matches
- See that the number of clusters you ask for changes what the algorithm finds
- Compare k-means, hierarchical clustering (single and Ward linkage), and DBSCAN on the same data
- Use PCA to picture the clusters and to name what separates them
- Report honestly what clustering can and cannot show

---
## 1. ASK: what is k-means really finding?
---

In Section 2.2 of the Module 9 notes, k-means with `k = 3` recovered the penguin species about 92% of the time, and every mistake confused Adelie with Chinstrap. We concluded that k-means found **size**, and that size happens to identify Gentoo cleanly.

That left a loose end: *which* Adelie and Chinstrap penguins were misplaced, and why those birds?

A hint comes from the people who collected the data. The title of their study (Gorman, Williams, and Fraser, 2014) begins **"Ecological sexual dimorphism"**: they were studying how males and females of the same species differ. So today's question is:

> **When an algorithm groups penguins by four body measurements, is it finding species, sex, or simply size?**

We will hide `species` and `sex` from every algorithm, cluster on the measurements alone, and only then compare.

#### **Activity 9.6 - Predict before you look**

Answer before running any code. There are no wrong predictions, only untested ones.

**A.** The data has three species, and each has males and females. If every species-and-sex combination were its own clump, how many clusters would there be?

**B.** From the Module 9 notes you know Gentoo are much bigger than the other two species. Within a species, which sex would you guess is larger? Which pair of groups do you expect to be hardest to tell apart?

**C.** Predict: will the elbow plot point to the number you gave in part A?

---
## 2. GET and EXPLORE: the data, with the labels set aside
---

In [ ]:
# RUN-TOGETHER  (setup)
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

raw = sns.load_dataset('penguins')
print('shape:', raw.shape)
print(raw.isna().sum())

Eleven penguins have no recorded `sex`, and two of those also have no measurements. Since we want to compare clusters with sex afterward, we keep only complete rows.

Then we **set the labels aside**. The algorithms will see only the four measurements in `X_pen`. We keep `species` and `sex` in the table only so we can check the clusters at the end, and we combine them into one column, `group`, for the comparison.

In [ ]:
# RUN-TOGETHER
penguins = raw.dropna().copy()
penguins['group'] = penguins['species'] + '-' + penguins['sex']
print('rows kept:', len(penguins))
print(penguins['group'].value_counts())

features = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
penguins[features].describe().round(1)

The `describe()` table repeats the lesson from Section 2.2: body mass has a standard deviation around 805 grams, while bill depth has one around 2 millimeters. Without scaling, body mass would decide almost every distance. So we scale first, and every method below uses the same scaled array.

In [ ]:
# RUN-TOGETHER
X_pen = StandardScaler().fit_transform(penguins[features])   # mean 0 and standard deviation 1 in every column
print(X_pen.shape)
print('column means:', X_pen.mean(axis=0).round(2))
print('column SDs  :', X_pen.std(axis=0).round(2))

---
## 3. MODEL I: k-means, asking for different numbers of clusters
---

#### **Class Example 9.3 - The same data, three different answers**

We already know the elbow plot for these penguins (Activity 9.2). Here are the numbers again.

In [ ]:
# RUN-TOGETHER
ks = range(1, 9)
wcss = [KMeans(n_clusters=k, n_init=10, random_state=0).fit(X_pen).inertia_ for k in ks]
print('WCSS :', np.round(wcss, 1))
print('drops:', np.round(-np.diff(wcss), 1))

The first drop dwarfs the rest, so the elbow points to `k = 2`. But nothing forces us to stop there. Clustering is a tool for asking questions, and different values of `k` ask different questions. We fit three versions and keep the labels of each.

In [ ]:
# RUN-TOGETHER
for k in [2, 3, 6]:
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(X_pen)
    penguins[f'k{k}'] = km.labels_          # new columns k2, k3, k6

print(pd.crosstab(penguins['k2'], penguins['species']))

With `k = 2`, the split is exactly **Gentoo vs everyone else**: 119 Gentoo in one cluster, all 146 Adelie and 68 Chinstrap in the other. That matches what the elbow told us. The biggest structure in the data is "Gentoo-sized" vs "not Gentoo-sized."

Now `k = 3`, the version from the Module 9 notes. This time we compare against `group`, which shows species and sex together.

In [ ]:
# RUN-TOGETHER
pd.crosstab(penguins['k3'], penguins['group'])

#### **Activity 9.7 - Who are the misfits?**

**A.** Find the cluster that holds most of the Chinstrap penguins. Which Adelie penguins are in it: males, females, or both? How many?

**B.** Find the Adelie cluster. Which Chinstrap penguins ended up there?

**C.** Complete the cell below to compare the Adelie males in the "Chinstrap" cluster with the Adelie males in the "Adelie" cluster. Which measurements differ most?

**D.** Put parts A to C together in two sentences: why did k-means make exactly these mistakes?

In [ ]:
# FILL-IN  (Activity 9.7, Part C)
adelie_males = penguins[(penguins['species'] == 'Adelie') & (penguins['sex'] == ____)]
adelie_males.groupby(____)[features].mean().round(1)

That suggests sex is part of what the measurements carry. If species and sex together make six groups, what happens if we simply **ask for six**?

In [ ]:
# RUN-TOGETHER
pd.crosstab(penguins['k6'], penguins['group'])

In [ ]:
# RUN-TOGETHER  (what each of the six clusters looks like)
penguins.groupby('k6')[features].mean().round(1)

#### **Activity 9.8 - Six clusters**

**A.** For each of the six clusters, write the species-and-sex group that makes up most of it.

**B.** Add up the counts in the cell of each cluster's main group. What fraction of the 333 penguins land in a cluster whose main group matches their own? Check your arithmetic with the cell below.

**C.** Use the cluster means to describe one pair of clusters that come from the same species. How do the two sexes differ?

**D.** The elbow plot did not point to six. Was choosing `k = 6` a discovery, or did we use information the algorithm was not supposed to have? Would you have found these six groups if you had never seen the species or sex labels?

In [ ]:
# FILL-IN  (Activity 9.8, Part B)
table = pd.crosstab(penguins['k6'], penguins['group'])
matched = table.____(axis=1).sum()        # the largest count in each row, added up
print('matched:', matched, 'of', len(penguins), '=', round(matched / len(penguins), 3))

---
## 4. MODEL II: hierarchical clustering, two ways of measuring "close"
---

Section 4 of the Module 9 notes built dendrograms with **single linkage**: the distance between two clusters is the distance between their *closest* pair of points. There are other rules. **Ward linkage** (described in Huber's *Clustering beyond the basics*) merges the two clusters whose union increases the within-cluster sum of squares the least, the same WCSS that k-means minimizes. Ward tends to build compact, similar-sized clusters; single linkage tends to build long chains.

#### **Class Example 9.4 - Same data, two linkages**

In [ ]:
# RUN-TOGETHER
Z_single = linkage(X_pen, method='single')
Z_ward = linkage(X_pen, method='ward')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
dendrogram(Z_single, truncate_mode='lastp', p=15, ax=axes[0])   # show only the last 15 merges
axes[0].set_title('Single linkage (last 15 merges)')
dendrogram(Z_ward, truncate_mode='lastp', p=15, ax=axes[1])
axes[1].set_title('Ward linkage (last 15 merges)')
for ax in axes:
    ax.set_xlabel('cluster (number of penguins in parentheses)')
    ax.set_ylabel('merge height')
plt.show()

print('single, last five merge heights:', Z_single[-5:, 2].round(2))
print('Ward,   last five merge heights:', Z_ward[-5:, 2].round(1))

In [ ]:
# RUN-TOGETHER  (cut each tree into 3 clusters, then into 6 with Ward)
penguins['single3'] = fcluster(Z_single, 3, criterion='maxclust')
penguins['ward3'] = fcluster(Z_ward, 3, criterion='maxclust')
penguins['ward6'] = fcluster(Z_ward, 6, criterion='maxclust')

print(pd.crosstab(penguins['single3'], penguins['species']))
print(pd.crosstab(penguins['ward3'], penguins['species']))

#### **Activity 9.9 - Reading the two trees**

**A.** Look at the single-linkage cut into 3 clusters. How big is the smallest cluster? What does that say about what single linkage does with an unusual penguin?

**B.** In the Ward dendrogram, the last merge is far above the others. Which two groups does it join? Where in the single-linkage dendrogram is there a similar gap?

**C.** Compare the Ward 3-cluster cut with the k-means 3-cluster crosstab from Section 3. Which method keeps all the Adelie penguins together? Which Chinstrap penguins does Ward misplace?

**D.** Use one line of code to compare `ward6` with `group`. Does Ward also find the six species-and-sex groups?

In [ ]:
# Activity 9.9, Part D: your one line here

---
## 5. MODEL III: DBSCAN, and the penguins that fit nowhere
---

DBSCAN does not take a number of clusters. It grows clusters from dense neighborhoods and labels anything left over as noise (`-1`).

In [ ]:
# RUN-TOGETHER
for eps in [0.3, 0.5, 0.8]:
    db = DBSCAN(eps=eps, min_samples=5).fit(X_pen)
    n_clusters = len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)
    print(f'eps={eps}  clusters={n_clusters}  noise={(db.labels_ == -1).sum()}')

#### **Activity 9.10 - Clusters and outliers**

**A.** Which `eps` gives a clean answer with very little noise? Complete the cell below with that value. Which groups does DBSCAN separate, and which does it merge?

**B.** Why does DBSCAN merge Adelie and Chinstrap even though k-means split them? (Think about chains of neighbors, as in the Module 9 crescents.)

**C.** Print the noise penguins. For each one, find the measurement that looks unusual for its species and sex, using the group means printed below. Would you call these birds errors, or just unusual? What would you need to know to decide?

In [ ]:
# FILL-IN  (Activity 9.10)
db = DBSCAN(eps=____, min_samples=5).fit(X_pen)
penguins['db'] = db.labels_
print(pd.crosstab(penguins['db'], penguins['species']))

print(penguins[penguins['db'] == ____][['species', 'sex', 'island'] + features])
print(penguins.groupby(['species', 'sex'])[features].mean().round(1))

---
## 6. Picturing the clusters with PCA
---

#### **Class Example 9.5 - What separates the groups?**

Four dimensions cannot be drawn, but two principal components can. The printed **loadings** tell us what each component measures; Sections 6.1 and 6.2 show how to read them.

In [ ]:
# RUN-TOGETHER
pca = PCA(n_components=2).fit(X_pen)
pcs = pca.transform(X_pen)
penguins['PC1'] = pcs[:, 0]
penguins['PC2'] = pcs[:, 1]

print('variance explained:', pca.explained_variance_ratio_.round(3), '| together:', round(pca.explained_variance_ratio_.sum(), 3))
print(pd.DataFrame(pca.components_.T, index=features, columns=['PC1', 'PC2']).round(2))

### 6.1 How to read a loadings table

Each column of the loadings table is a **recipe** for one component. To get a penguin's PC1 score, take each of its *scaled* measurements, multiply by that measurement's loading, and add:

$$	ext{PC1} = 0.45\,z_{	ext{bill length}} - 0.40\,z_{	ext{bill depth}} + 0.58\,z_{	ext{flipper}} + 0.55\,z_{	ext{mass}}$$

Here $z$ is the scaled value from `X_pen`: how many standard deviations the penguin sits above (positive) or below (negative) the average penguin. A penguin that is exactly average on every measurement has all $z = 0$, so it scores 0 on every component.

Four rules for reading the numbers:

1. **Sign.** A positive loading means *larger* values of that measurement raise the score. A negative loading means larger values *lower* it. On PC1, a long flipper raises the score and a deep bill lowers it.
2. **Size.** The larger the loading (ignoring its sign), the more that measurement moves the score. A loading near 0 means the measurement hardly matters for that component: flipper length has a PC2 loading of 0.01, so flipper length plays essentially no role in PC2.
3. **Shares.** The squared loadings in each column add to 1. On PC1, $0.58^2 pprox 0.33$ and $0.55^2 pprox 0.30$, so flipper length and body mass together carry about two thirds of PC1's weight. No single measurement dominates PC1; on PC2, bill depth ($0.80^2 pprox 0.63$) carries most of the weight.
4. **The overall sign is arbitrary.** If every sign in a column were flipped, the component would describe the same direction, just pointing the other way, and every score would change sign. Different software versions can return either. So read the **pattern** within a column (which measurements move together, which move opposite), not whether PC1 happens to be positive.

A useful consequence of rule 1: a **negative loading times a below-average value is a positive contribution**. A penguin with a shallow bill (negative $z$) gets a *higher* PC1 score from its bill depth.

In [ ]:
# RUN-TOGETHER  (check rule 3: squared loadings add to 1 in each component)
print((pca.components_ ** 2).sum(axis=1).round(3))

### 6.2 A worked example: one penguin's scores

Let's follow the recipe for one bird, the penguin at position 197 of our table (counting from 0, as `.iloc` and `X_pen` both do). The `Name:` line in the output shows its original row label, 203, which differs because we dropped the incomplete rows.

In [ ]:
# RUN-TOGETHER
i = 197
print(penguins.iloc[i][['species', 'sex'] + features], '\n')

z = X_pen[i]                                    # this penguin's four scaled values
recipe = pd.DataFrame({'scaled value z': z,
                       'PC1 loading': pca.components_[0],
                       'PC1 loading x z': pca.components_[0] * z,
                       'PC2 loading': pca.components_[1],
                       'PC2 loading x z': pca.components_[1] * z},
                      index=features)
print(recipe.round(2), '\n')

print('PC1 by hand:', round((pca.components_[0] * z).sum(), 2), '   PCA says:', round(pcs[i, 0], 2))
print('PC2 by hand:', round((pca.components_[1] * z).sum(), 2), '   PCA says:', round(pcs[i, 1], 2))

Reading the table row by row:

- He is a **Chinstrap male** with a long bill (51.4 mm, $z = 1.36$), a deep bill (19.0 mm, $z = 0.93$), an average flipper ($z = 0.00$), and a slightly light body ($z = -0.32$).
- **PC1 is about 0.07, close to 0**, the score of a perfectly average penguin. His long bill pushes PC1 up by 0.62, but his deep bill pulls it down by 0.37 (a positive $z$ times the negative loading), and his light body pulls it down another 0.18. The pieces nearly cancel: on the overall size axis he is a typical penguin.
- **PC2 is about 1.53, high.** On PC2 both bill measurements have positive loadings, so his long bill (+0.81) and deep bill (+0.74) add up instead of canceling, and his flipper and body hardly count.

So the two scores tell a short story: *average body size, unusually large bill.* That is exactly what distinguishes Chinstrap males in the PCA picture below, where they sit at the top.

#### **Activity 9.11 - Read a penguin's scores**

The penguin at position 257 is a Gentoo male (50.5 mm bill length, 15.9 mm bill depth, 222 mm flipper, 5550 g). The penguin at position 128 is an Adelie female (38.1 mm, 17.6 mm, 187 mm, 3425 g).

**A.** Before running any code, use the loadings and the four rules to predict whether each bird's PC1 score is positive or negative, and whether it is large or near 0. Explain in one sentence each.

**B.** Complete the cell below to compute the Gentoo male's contributions and scores. Which single measurement contributes most to his PC1? His bill depth is below average: does it raise or lower his PC1, and why?

**C.** Change `i` to 128 and run the cell again. Describe the Adelie female in one sentence, as we did for the Chinstrap male.

**D.** Suppose a new version of scikit-learn returned the PC1 loadings with every sign flipped. What would happen to the three birds' PC1 scores, and would your descriptions change?

In [ ]:
# FILL-IN  (Activity 9.11)
i = 257
print(penguins.iloc[i][['species', 'sex'] + features], '\n')

z = X_pen[____]
contrib1 = pca.components_[0] * ____
contrib2 = pca.components_[1] * z
print(pd.DataFrame({'z': z, 'PC1 part': contrib1, 'PC2 part': contrib2}, index=features).round(2))
print('PC1 =', round(contrib1.____(), 2), '  PC2 =', round(contrib2.sum(), 2))

### 6.3 Naming the components, and picturing the clusters

In [ ]:
# RUN-TOGETHER
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(data=penguins, x='PC1', y='PC2', hue='k6', palette='tab10', s=25, ax=axes[0])
axes[0].set_title('k-means with k = 6 (no labels used)')
# hue_order lists the groups in the order of their matching k-means clusters, so colors line up across the panels
order = ['Adelie-Male', 'Gentoo-Male', 'Adelie-Female', 'Gentoo-Female', 'Chinstrap-Female', 'Chinstrap-Male']
sns.scatterplot(data=penguins, x='PC1', y='PC2', hue='group', hue_order=order, palette='tab10', s=25, ax=axes[1])
axes[1].set_title('The hidden labels: species and sex')
plt.show()

Putting the rules together to **name** each component:

- **PC1** (about 69% of the variation): flipper length, body mass, and bill length load positively with similar weights, and bill depth loads negatively. High PC1 means a big, long-flippered bird with a long but shallow bill. It is roughly an **overall size** axis, tilted toward the Gentoo body plan, which is why Gentoo sit on the right.
- **PC2** (about 19%): bill depth and bill length both load positively, and flipper and body mass load near 0. High PC2 means a large bill *for a penguin of that body size*. It is roughly a **bill size** axis.

Naming components is interpretation; the loadings are the evidence. A good habit is to check a name against a few individual birds, as in Section 6.2.

> **Discuss.** In the right-hand picture, find the three pairs of species-and-sex groups. Within each species, are males and females separated mostly along PC1, mostly along PC2, or along both? Where do the Adelie males and the Chinstrap females come closest? Compare with the misfits you found in Activity 9.7.

---
## 7. One species at a time
---

So far sex has competed with species for the algorithm's attention. A cleaner test of the researchers' question removes species from the picture: within a single species, do two clusters line up with sex?

#### **Class Example 9.6 - Clustering inside each species**

The loop below repeats the same three steps for each species: keep that species' rows, scale, and fit k-means with `k = 2`. (We rescale inside each species, because the spread of each measurement is different once Gentoo are removed.)

In [ ]:
# RUN-TOGETHER
for sp in ['Adelie', 'Chinstrap', 'Gentoo']:
    one = penguins[penguins['species'] == sp]
    X_one = StandardScaler().fit_transform(one[features])
    labels = KMeans(n_clusters=2, n_init=10, random_state=0).fit(X_one).labels_
    table = pd.crosstab(labels, one['sex'])
    print(sp, '| matched:', table.max(axis=1).sum(), 'of', len(one))
    print(table, '\n')

> **Discuss.** In every species, two clusters line up with sex for close to 9 out of 10 birds. What does that say about the researchers' topic, sexual dimorphism? Suppose a field biologist wanted to guess the sex of an unlabeled penguin from these four measurements. Why would a **supervised** method, trained on birds whose sex is known (Module 10), be a better tool than clustering?

---
## 8. COMMUNICATE: what did we learn?
---

| Method | Setting | What it found |
| --- | --- | --- |
| k-means | k = 2 (the elbow) | Gentoo vs everyone else, exactly |
| k-means | k = 3 | the three species, about 92%; mistakes at the large-Adelie-male / small-Chinstrap-female boundary |
| k-means | k = 6 | the six species-and-sex groups, about 90% |
| hierarchical, single | 3 clusters | one long chain of Adelie + Chinstrap and a single isolated penguin |
| hierarchical, Ward | 2 and 6 clusters | Gentoo vs the rest (a very large gap); six groups about 86% |
| DBSCAN | eps = 0.8 | Gentoo vs Adelie + Chinstrap, plus 5 unusual penguins |
| PCA | 2 components | about 88% of the variation; PC1 is overall size, PC2 is bill size |
| k-means within each species | k = 2 | sex, about 88% to 90% in every species |

#### **Activity 9.12 - Answer the question**

**A.** In 4 to 5 sentences, answer the question from Section 1 for a reader who has never run a clustering algorithm: *is clustering finding species, sex, or size?* Quote at least two numbers from the table above.

**B.** Give two cautions you would attach to your answer.

**C.** Which single method would you show the researchers first, and why?

> **Discuss.** This case study had an answer key hidden in the data: we could check every cluster against species and sex. In most real unsupervised problems (customer segments, river conditions, documents) there is no such key. Which of today's checks would still be available without labels, and which would not?